In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score
from sklearn.model_selection import StratifiedKFold
from pyod.models.deep_svdd import DeepSVDD
import sys
import scrapbook as sb

sys.path.append('..')
from utils import reshape_to_numpy, interpolate_missing_values, denoise_data, compute_spectrograms, time_avg_pooling

In [2]:
seed = 1
sampling_rate = 10
nperseg=64
noverlap=32
accel_cutoff = 0.4
accel_order = 30
gyro_cutoff = 0.6
gyro_order = 20
batch_size = 32
epochs = 100
hidden_neurons = [128, 64, 32]
n_splits = 5

In [3]:
# Parameters
seed = 27


In [4]:
rng = np.random.RandomState(seed)

In [5]:
data = pd.read_parquet("../data/GBG500.parquet")
data

,ride_id,time_index,ax,ay,az,rx,ry,rz
0,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,0,-2.512796,-9.385012,-1.053078,-0.009155,0.009155,-0.201416
1,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,100,-2.491268,-9.385012,-1.079390,-0.036621,0.027465,-0.155639
2,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,200,-2.534324,-9.382022,-1.030952,0.036621,0.027465,-0.073242
3,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,300,-2.488278,-9.383218,-1.091948,-0.045776,0.036621,-0.183105
4,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,400,-2.483494,-9.382022,-1.100320,-0.045776,0.027465,-0.192260
...,...,...,...,...,...,...,...,...
1529542,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241300,0.459862,-9.140430,-3.318900,1.556396,-0.091552,0.274658
1529543,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241400,0.455676,-9.140430,-3.321292,1.583861,-0.119018,0.274658
1529544,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241500,0.459862,-9.139832,-3.317704,1.583861,-0.109863,0.274658
1529545,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241600,0.456274,-9.142224,-3.321292,1.574707,-0.137329,0.283813


In [6]:
labels = pd.read_csv("../data/GBG500_labels.csv")
labels.columns = labels.columns.str.lower()
ride_order_df = pd.DataFrame({"ride_id": data["ride_id"].unique()})
labels_sorted = ride_order_df.merge(
    labels,
    on="ride_id",
    how="left"
)
labels_sorted

,ride_id,label
0,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,Safe
1,00d223cb7aecc9c0cc5871e32a6c45027a068eba07c572...,Reckless
2,02288a4aeca044203394e982e76b87818021dea2b34df9...,Safe
3,0236bdcf13d473ea24d97f5eaeea459f257dffac005694...,Bad weather
4,0238e5dd85143f1b6c59b200bc32f14361b8fc84c51a87...,Safe
...,...,...
495,fe5d7f84d692bbccad8bd566a3ad5c3108fa9f90acd671...,Safe
496,fe67169632306d4668b2affedef510df2cb43e2fb41220...,Safe
497,fee44667fdc8ab4995c702fc3bd36178de0b1ca2c64687...,Safe
498,ff79b2e945b93c2a5efc3a06365267b3a160e7849ba57d...,Safe


In [7]:
data_np = reshape_to_numpy(
    data,
    features = ["ax", "ay", "az", "rx", "ry", "rz"],
    max_timestamps = 4800
)

data_np_clean = interpolate_missing_values(
    data_np,
    method='linear',
    limit=None
)

data_np_clean = denoise_data(
    data=data_np_clean,
    accel_indices=[0, 1, 2],
    gyro_indices=[3, 4, 5],
    accel_cutoff=accel_cutoff,
    accel_order=accel_order,
    gyro_cutoff=gyro_cutoff,
    gyro_order=gyro_order,
)

In [8]:
# Compute spectrograms for all rides
spectrograms_array = compute_spectrograms(data_np_clean, sampling_rate, nperseg, noverlap)
spectrograms_array.shape

(500, 149, 6, 33)

In [9]:
# Aggregate spectrograms using time-averaged pooling
X_feat = time_avg_pooling(spectrograms_array)
X_feat.shape

(500, 198)

In [10]:
# 5-fold stratified CV with Deep SVDD
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=rng.randint(1000))
y_true = (labels_sorted['label'] == 'Reckless').astype(int).values
anomaly_scores = np.full(len(y_true), np.nan)

for fold, (train_idx, test_idx) in enumerate(skf.split(X_feat, y_true)):
    sc = StandardScaler()
    X_train = sc.fit_transform(X_feat[train_idx])
    X_test = sc.transform(X_feat[test_idx])

    np.random.seed(rng.randint(1000))
    model = DeepSVDD(
        n_features=X_train.shape[1],
        hidden_neurons=hidden_neurons,
        epochs=epochs,
        batch_size=batch_size,
        random_state=rng.randint(1000),
    )
    model.fit(X_train)
    anomaly_scores[test_idx] = model.decision_function(X_test)
    print(f"Fold {fold+1}/{n_splits} done")

Epoch 1/100, Loss: 1.8433543741703033
Epoch 2/100, Loss: 1.8853118792176247
Epoch 3/100, Loss: 1.9787829592823982
Epoch 4/100, Loss: 1.8947486951947212
Epoch 5/100, Loss: 1.955077700316906
Epoch 6/100, Loss: 1.831634022295475
Epoch 7/100, Loss: 1.8451563715934753
Epoch 8/100, Loss: 1.850570522248745
Epoch 9/100, Loss: 1.808224618434906
Epoch 10/100, Loss: 1.9008816704154015
Epoch 11/100, Loss: 1.7804469540715218
Epoch 12/100, Loss: 1.810337856411934
Epoch 13/100, Loss: 1.741174928843975
Epoch 14/100, Loss: 1.7902375981211662
Epoch 15/100, Loss: 1.7521952390670776
Epoch 16/100, Loss: 1.8205187022686005
Epoch 17/100, Loss: 1.8795600533485413
Epoch 18/100, Loss: 1.743560716509819


Epoch 19/100, Loss: 1.905036374926567
Epoch 20/100, Loss: 1.9592646360397339
Epoch 21/100, Loss: 1.9113254696130753
Epoch 22/100, Loss: 1.807112105190754
Epoch 23/100, Loss: 1.8643719479441643
Epoch 24/100, Loss: 1.8001838028430939
Epoch 25/100, Loss: 1.8521501049399376
Epoch 26/100, Loss: 1.8964171335101128
Epoch 27/100, Loss: 1.8800579011440277
Epoch 28/100, Loss: 1.8786183297634125
Epoch 29/100, Loss: 1.8575567528605461
Epoch 30/100, Loss: 1.8080930188298225
Epoch 31/100, Loss: 1.8355068042874336
Epoch 32/100, Loss: 1.9002751633524895
Epoch 33/100, Loss: 1.8503069877624512
Epoch 34/100, Loss: 1.7773936428129673
Epoch 35/100, Loss: 1.873946525156498
Epoch 36/100, Loss: 1.7785071954131126


Epoch 37/100, Loss: 1.8124465756118298
Epoch 38/100, Loss: 1.9069305211305618
Epoch 39/100, Loss: 1.8744421005249023
Epoch 40/100, Loss: 1.822204701602459
Epoch 41/100, Loss: 1.9099212065339088
Epoch 42/100, Loss: 1.8744367212057114
Epoch 43/100, Loss: 1.9329049289226532
Epoch 44/100, Loss: 1.8002836182713509
Epoch 45/100, Loss: 1.799048125743866
Epoch 46/100, Loss: 1.858989231288433
Epoch 47/100, Loss: 1.8212238252162933
Epoch 48/100, Loss: 1.848849967122078
Epoch 49/100, Loss: 1.905015490949154
Epoch 50/100, Loss: 1.9393284544348717
Epoch 51/100, Loss: 1.854333221912384
Epoch 52/100, Loss: 1.850904032588005
Epoch 53/100, Loss: 1.8772659376263618
Epoch 54/100, Loss: 1.8049089014530182
Epoch 55/100, Loss: 2.0218792632222176


Epoch 56/100, Loss: 1.793954737484455
Epoch 57/100, Loss: 1.8166585341095924
Epoch 58/100, Loss: 1.9039753451943398
Epoch 59/100, Loss: 1.8067908734083176
Epoch 60/100, Loss: 1.8246879875659943
Epoch 61/100, Loss: 1.896608680486679
Epoch 62/100, Loss: 1.9295902848243713
Epoch 63/100, Loss: 1.8999335169792175
Epoch 64/100, Loss: 1.9068838953971863
Epoch 65/100, Loss: 1.7999622002243996
Epoch 66/100, Loss: 1.7611751854419708
Epoch 67/100, Loss: 1.8305535912513733
Epoch 68/100, Loss: 1.954644188284874
Epoch 69/100, Loss: 1.9048377647995949
Epoch 70/100, Loss: 1.8391265720129013
Epoch 71/100, Loss: 1.98176359385252
Epoch 72/100, Loss: 1.9075827300548553


Epoch 73/100, Loss: 1.8259423300623894
Epoch 74/100, Loss: 1.9082768931984901
Epoch 75/100, Loss: 1.8879837840795517
Epoch 76/100, Loss: 1.9309136942029
Epoch 77/100, Loss: 1.8787061050534248
Epoch 78/100, Loss: 1.7051602005958557
Epoch 79/100, Loss: 1.8543005250394344
Epoch 80/100, Loss: 1.8074841648340225
Epoch 81/100, Loss: 1.8606985360383987
Epoch 82/100, Loss: 2.0055016949772835
Epoch 83/100, Loss: 1.7973762527108192
Epoch 84/100, Loss: 1.787547044456005
Epoch 85/100, Loss: 1.8165747225284576
Epoch 86/100, Loss: 1.7182091102004051
Epoch 87/100, Loss: 1.8568232357501984
Epoch 88/100, Loss: 1.7567724138498306
Epoch 89/100, Loss: 1.7998643219470978
Epoch 90/100, Loss: 1.8578835800290108


...

Epoch 24/100, Loss: 2.2229763194918633
Epoch 25/100, Loss: 2.147872284054756
Epoch 26/100, Loss: 2.3415169939398766
Epoch 27/100, Loss: 2.575695261359215
Epoch 28/100, Loss: 2.3677247539162636
Epoch 29/100, Loss: 2.248223390430212
Epoch 30/100, Loss: 2.0744268372654915
Epoch 31/100, Loss: 2.50611924380064
Epoch 32/100, Loss: 2.296286754310131
Epoch 33/100, Loss: 2.3294494971632957
Epoch 34/100, Loss: 2.145040661096573
Epoch 35/100, Loss: 2.263757459819317
Epoch 36/100, Loss: 1.9174571707844734
Epoch 37/100, Loss: 2.182734802365303
Epoch 38/100, Loss: 2.4113649278879166
Epoch 39/100, Loss: 2.503052443265915
Epoch 40/100, Loss: 2.3715448677539825
Epoch 41/100, Loss: 2.225120611488819


Epoch 42/100, Loss: 2.373017944395542
Epoch 43/100, Loss: 2.136720672249794
Epoch 44/100, Loss: 2.316695623099804
Epoch 45/100, Loss: 2.3168369457125664
Epoch 46/100, Loss: 2.5965797379612923
Epoch 47/100, Loss: 2.407640539109707
Epoch 48/100, Loss: 2.360546000301838
Epoch 49/100, Loss: 2.1604557558894157
Epoch 50/100, Loss: 2.243644416332245
Epoch 51/100, Loss: 2.4743726328015327
Epoch 52/100, Loss: 2.3999196514487267
Epoch 53/100, Loss: 2.8606544360518456
Epoch 54/100, Loss: 2.3968233913183212
Epoch 55/100, Loss: 2.2201817110180855
Epoch 56/100, Loss: 2.1270938143134117
Epoch 57/100, Loss: 2.158092126250267
Epoch 58/100, Loss: 2.302395962178707
Epoch 59/100, Loss: 2.174810141324997


Epoch 60/100, Loss: 2.339529648423195
Epoch 61/100, Loss: 2.200187101960182
Epoch 62/100, Loss: 2.408475488424301
Epoch 63/100, Loss: 2.2265926226973534
Epoch 64/100, Loss: 2.212424397468567
Epoch 65/100, Loss: 2.341366358101368
Epoch 66/100, Loss: 2.2016477808356285
Epoch 67/100, Loss: 2.2809689790010452
Epoch 68/100, Loss: 2.1950458213686943
Epoch 69/100, Loss: 2.1537389308214188
Epoch 70/100, Loss: 2.522979326546192
Epoch 71/100, Loss: 2.316500701010227
Epoch 72/100, Loss: 2.0862099900841713
Epoch 73/100, Loss: 2.3312597796320915
Epoch 74/100, Loss: 2.313226506114006
Epoch 75/100, Loss: 2.5607911944389343
Epoch 76/100, Loss: 2.1204611510038376
Epoch 77/100, Loss: 2.335120625793934


Epoch 78/100, Loss: 2.117955021560192
Epoch 79/100, Loss: 2.275103021413088
Epoch 80/100, Loss: 2.242228388786316
Epoch 81/100, Loss: 2.2467033714056015
Epoch 82/100, Loss: 2.3493715077638626
Epoch 83/100, Loss: 2.152605175971985
Epoch 84/100, Loss: 1.9820669740438461
Epoch 85/100, Loss: 2.4154499173164368
Epoch 86/100, Loss: 2.354188449680805
Epoch 87/100, Loss: 2.3065706565976143
Epoch 88/100, Loss: 2.4994298443198204
Epoch 89/100, Loss: 2.3770902678370476
Epoch 90/100, Loss: 2.2672991827130318
Epoch 91/100, Loss: 2.282999560236931
Epoch 92/100, Loss: 2.3152543380856514
Epoch 93/100, Loss: 2.0337583646178246
Epoch 94/100, Loss: 2.2446207851171494
Epoch 95/100, Loss: 2.0817426592111588


Epoch 96/100, Loss: 2.292320504784584
Epoch 97/100, Loss: 2.2047043219208717
Epoch 98/100, Loss: 2.172125168144703
Epoch 99/100, Loss: 2.207752250134945
Epoch 100/100, Loss: 2.374122068285942
Fold 5/5 done


In [11]:
ap = average_precision_score(y_true, anomaly_scores)
print(f"Deep SVDD AP (5-fold CV) = {ap:.4f}")
sb.glue("GBG500_ap_spectral_deep_svdd", float(ap))

Deep SVDD AP (5-fold CV) = 0.5691
